<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

## 🌌 Week 12 — Generative AI & the Modern AI Landscape: Diffusion, LLMs, and Beyond
### CP020003 Artificial Intelligence 2026 — In-Class Notebook (Final Lecture)

Last week we asked a model **"what is in this image?"** — classification, boxes, masks, depth, pose. Every one of those is *discriminative*: the model looks at something that already exists and describes it.

This week we flip the question. We ask a model to **make something that didn't exist a second ago.**

That's **Generative AI** — and in 2026 it comes in two dominant flavors that think in completely different ways:

| | 🧠 Large Language Models (LLM) | 🎨 Diffusion Models |
|---|---|---|
| Data type | Text (discrete tokens) | Images / audio (continuous signals) |
| How it generates | **Autoregressive** — one token at a time, left → right, never revising | **Iterative denoising** — refines the *entire* canvas together, many times |
| Training signal | "Predict the next token" | "Predict the noise that was added, so we can remove it" |
| Analogy | Writing a sentence word by word, committing as you go | Sculpting a statue out of a block of noise, refining the whole thing pass after pass |
| 2026 frontier | Bigger context, tool use, reasoning chains | Fewer steps (distillation), new math (flow matching) |

And there's a third thread running through everything below: **modern AI is diffusion getting radically faster.** Techniques like *Adversarial Diffusion Distillation* (SD-Turbo) and *Latent Consistency Models* (LCM) compress 20–50 denoising steps down to **1–4** — the difference between "wait 10 seconds" and "instant." We'll see this with our own eyes in Section 6.

### 🗺️ Today's roadmap
1. **LLM vs Diffusion** — watch each one "think" 🧠
2. **Text → Image** — turn a prompt into art 🎨
3. **Hyperparameters + a real metric** — steps, guidance scale, and CLIP Score 📊
4. **Image → Text** — caption what we just created 📝
5. **The Round Trip** — image → text → image, and how much meaning survives 🔁
6. **Inpainting** — edit *only* part of an image 🖌️
7. **Beyond Classic Diffusion** — 1-step "Turbo" generation ⚡
8. **Wrap-up + exercises** 🎓

> 💡 **Runtime:** `Runtime → Change runtime type → T4 GPU` (free tier). We use small/distilled models and load only **one model at a time**, freeing GPU memory between sections — that's how this whole tour fits on a free 15 GB T4. Every generation below uses a handful of images and steps on purpose; this class is about *understanding the pipeline*, not chasing state-of-the-art art.

## 0. Setup 🔧

Colab ships `torch` and `matplotlib` already. We add `diffusers` (the Hugging Face library behind Stable Diffusion, SD-Turbo, inpainting, and friends) and `transformers` (LLMs, BLIP captioning, CLIP scoring) — one consistent ecosystem for almost every generative task that exists today.

Because we're moving through **six different models** on a single free GPU, we'll define a `free_memory()` helper up front and call it every time we're done with a model.

In [ ]:
# Write the name of your modern ai library here

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageDraw

SEED = # Write your lucky number here
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("⚠️  No GPU detected — go to Runtime > Change runtime type > T4 GPU for a big speedup.")
else:
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def free_memory(*names):
    for name in names:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def show_images(images, titles=None, cols=None, figsize_per=3.2):
    n = len(images)
    cols = cols or n
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per * cols, figsize_per * rows))
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i])
            if titles is not None:
                ax.set_title(titles[i], fontsize=9)
            ax.axis("off")
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

## 1. Two Families of Generative AI: LLMs vs Diffusion 🧠

Before we touch images, let's actually *see* the difference the table above describes.

An LLM is **autoregressive**: it picks one token, appends it to the sequence, and picks the next one conditioned on everything so far. It can never go back and change a word it already committed to — if it wrote itself into a corner, it has to talk its way out, not undo.

We'll load a small, modern instruction-tuned model — **Qwen2.5-0.5B-Instruct** (500M parameters, easily fits a free T4) — and print each token the instant it's generated.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_ID = # Write your modern ai name here
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16).to(device)
llm.eval()
print("Loaded:", LLM_ID)

### 1.1 Watch it think, one token at a time

In [ ]:
def stream_generate(prompt, max_new_tokens=40):
    messages = [{"role": "user", "content": prompt}]

    # Get the actual input_ids tensor from the BatchEncoding
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).input_ids.to(device)

    generated = input_ids

    print(f"Prompt: {prompt}\n")
    print("Generating: ", end="", flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Run the model
            out = llm(generated)

            # Get logits for the last token
            next_token_logits = out.logits[:, -1, :]

            # Greedy decoding: choose the highest-probability token
            next_token = torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True
            )

            # Add the new token to the sequence
            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

            # Decode and print the new token
            piece = tokenizer.decode(
                next_token[0],
                skip_special_tokens=True
            )

            print(piece, end="", flush=True)

            # Stop if EOS token is generated
            if (
                tokenizer.eos_token_id is not None
                and next_token.item() == tokenizer.eos_token_id
            ):
                break

    print("\n")

    # Return only the newly generated text
    generated_text = tokenizer.decode(
        generated[0][input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return generated_text


# Test it
_ = stream_generate(
    "In one sentence, what makes diffusion models different from language models?"
)

**Notice:** the model committed to "In" and never reconsidered it, no matter what it wrote 30 tokens later. It only ever moves *forward*.

Diffusion works the opposite way: it starts with a canvas of **pure noise** and, at every one of its steps, looks at and revises the **whole image at once** — there's no "left" or "right," just successive full-image refinements. Let's watch that happen next.

In [ ]:
free_memory("llm", "tokenizer")
print("LLM freed from GPU memory ✅")

## 2. Text → Image: Turning Words into Art 🎨

**Task:** given a text prompt, generate a brand-new image that matches it. This is the flagship generative-AI demo — and the one your students will remember.

We'll use **Stable Diffusion v1.5**, the workhorse model behind most diffusion tutorials, loaded in half precision (`float16`) so it comfortably fits a free T4.

In [ ]:
from diffusers import StableDiffusionPipeline

SD_ID =  # Write your modern ai name here
sd_pipe = StableDiffusionPipeline.from_pretrained(
    SD_ID, torch_dtype=torch.float16, safety_checker=None
).to(device)
sd_pipe.set_progress_bar_config(disable=True)
print("Loaded:", SD_ID)

### 2.1 Generate art from a prompt

In [ ]:
prompts = [
    #  # Write your promts here
]

gen_images = []
for p in prompts:
    generator = torch.Generator(device=device).manual_seed(SEED)
    image = sd_pipe(p, num_inference_steps=25, guidance_scale=7.5, generator=generator).images[0]
    gen_images.append(image)

show_images(gen_images, titles=[p[:35] + "..." for p in prompts], cols=3)

### 2.2 Hyperparameter Tuning: steps & guidance scale ⚙️

Two knobs control almost every diffusion generation:

- **`num_inference_steps`** — how many denoising passes the model gets. More steps = more chances to refine detail, with diminishing returns and linearly more compute.
- **`guidance_scale`** (classifier-free guidance) — how strongly the model is pushed to follow the *text prompt* versus generating a "free" image. Low (~1) = loosely related, ignores the prompt more; high (~15+) = follows the prompt closely but can look over-saturated or artifacted; ~7–8 is the usual sweet spot.

Let's sweep both and *see* the trade-off directly, with the seed fixed so only the hyperparameters change.

In [ ]:
sweep_prompt = "a small robot watering a bonsai tree, studio lighting, highly detailed"
steps_list = [5, 15, 30]
guidance_list = [1.5, 7.5, 15.0]

sweep_images, sweep_titles, sweep_records = [], [], []
for steps in steps_list:
    for guidance in guidance_list:
        generator = torch.Generator(device=device).manual_seed(SEED)
        t0 = time.time()
        img = sd_pipe(
            sweep_prompt, num_inference_steps=steps, guidance_scale=guidance, generator=generator
        ).images[0]
        elapsed = time.time() - t0
        sweep_images.append(img)
        sweep_titles.append(f"steps={steps}, cfg={guidance}")
        sweep_records.append({"steps": steps, "guidance_scale": guidance, "seconds": elapsed, "image": img})

show_images(sweep_images, titles=sweep_titles, cols=len(guidance_list))

### 2.3 Evaluation Metric: CLIP Score 📊

Looking at a grid of images and going "yeah, that one looks better" isn't a metric — it's a vibe. Generative image models are usually scored with **CLIP Score**: encode the prompt and the image into the same embedding space with [CLIP](https://openai.com/research/clip), and measure their cosine similarity. Higher = the image better matches what the text asked for.

$$\text{CLIPScore}(I, T) = 100 \cdot \max(\cos(\text{CLIP}_{img}(I), \text{CLIP}_{txt}(T)),\ 0)$$

This gives us a real, comparable number for every cell in the hyperparameter grid above.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = # Write your modern ai name here
clip_model = CLIPModel.from_pretrained(CLIP_ID).to(device)
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip_model.eval()

@torch.no_grad()
def clip_score(image, text):
    inputs = clip_processor(text=[text], images=[image], return_tensors="pt", padding=True).to(device)
    outputs = clip_model(**inputs)
    img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
    txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
    sim = (img_emb @ txt_emb.T).item()
    return max(sim, 0) * 100

for rec in sweep_records:
    rec["clip_score"] = clip_score(rec["image"], sweep_prompt)

sweep_df = pd.DataFrame(sweep_records)[["steps", "guidance_scale", "seconds", "clip_score"]]
sweep_df

In [ ]:
pivot = sweep_df.pivot(index="steps", columns="guidance_scale", values="clip_score")
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel("guidance_scale"); ax.set_ylabel("num_inference_steps")
ax.set_title("CLIP Score across hyperparameters")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.1f}", ha="center", va="center", color="white", fontsize=9)
plt.colorbar(im, label="CLIP score")
plt.tight_layout()
plt.show()

print("Best combo:", sweep_df.loc[sweep_df.clip_score.idxmax(), ["steps", "guidance_scale", "clip_score"]].to_dict())

## 3. Image → Text: Describing What We Made 📝

**Task:** the reverse direction — given an image, generate a caption in natural language. We'll use **BLIP**, a small, fast image-captioning model, and run it on the art we generated in Section 2.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

CAPTION_ID =  # Write your modern ai name here
caption_processor = BlipProcessor.from_pretrained(CAPTION_ID)
caption_model = BlipForConditionalGeneration.from_pretrained(CAPTION_ID, torch_dtype=torch.float16).to(device)

@torch.no_grad()
def caption_image(image):
    inputs = caption_processor(image, return_tensors="pt").to(device, torch.float16)
    out = caption_model.generate(**inputs, max_new_tokens=30)
    return caption_processor.decode(out[0], skip_special_tokens=True)

captions = [caption_image(img) for img in gen_images]
for prompt, cap in zip(prompts, captions):
    print(f"Original prompt : {prompt}")
    print(f"BLIP's caption  : {cap}\n")

show_images(gen_images, titles=[c[:40] for c in captions], cols=3)

## 4. The Round Trip: Image → Text → Image 🔁

Here's a fun way to test how much meaning actually survives a translation between modalities:

1. Start with a prompt → generate **Image A**.
2. Caption **Image A** with BLIP → get **Prompt B** (the model's own words for what it made).
3. Feed **Prompt B** back into Stable Diffusion → generate **Image C**.
4. Compare A and C. Did the "telephone game" between an image model and a language model preserve the idea?

We'll measure it with CLIP two ways: **text-similarity** (Prompt A vs Prompt B) and **image-similarity** (Image A vs Image C, using CLIP's image encoder for both).

In [ ]:
# Round-trip generation
roundtrip_prompt_A = prompts[0]
image_A = gen_images[0]
prompt_B = captions[0]

generator = torch.Generator(device=device).manual_seed(SEED)

image_C = sd_pipe(
    prompt_B,
    num_inference_steps=25,
    guidance_scale=7.5,
    generator=generator
).images[0]

show_images(
    [image_A, image_C],
    titles=[
        f"A: '{roundtrip_prompt_A[:30]}...'",
        f"C: '{prompt_B[:30]}...'"
    ],
    cols=2
)


# ---------------------------------------------------------
# CLIP image similarity
# ---------------------------------------------------------
@torch.no_grad()
def clip_image_similarity(img1, img2):
    inputs = clip_processor(
        images=[img1, img2],
        return_tensors="pt"
    ).to(device)

    outputs = clip_model.get_image_features(**inputs)

    # Some Transformers versions return a tensor,
    # while others return a model output object.
    if hasattr(outputs, "pooler_output"):
        embeds = outputs.pooler_output
    elif hasattr(outputs, "last_hidden_state"):
        embeds = outputs.last_hidden_state[:, 0]
    else:
        embeds = outputs

    embeds = embeds / embeds.norm(dim=-1, keepdim=True)

    return (embeds[0] @ embeds[1]).item()


# ---------------------------------------------------------
# CLIP text similarity
# ---------------------------------------------------------
@torch.no_grad()
def clip_text_similarity(text1, text2):
    inputs = clip_processor(
        text=[text1, text2],
        return_tensors="pt",
        padding=True
    ).to(device)

    outputs = clip_model.get_text_features(**inputs)

    # Some Transformers versions return a tensor,
    # while others return a model output object.
    if hasattr(outputs, "pooler_output"):
        embeds = outputs.pooler_output
    elif hasattr(outputs, "last_hidden_state"):
        embeds = outputs.last_hidden_state[:, 0]
    else:
        embeds = outputs

    embeds = embeds / embeds.norm(dim=-1, keepdim=True)

    return (embeds[0] @ embeds[1]).item()


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------
print(f"Prompt A: {roundtrip_prompt_A}")
print(f"Prompt B (BLIP's caption of Image A): {prompt_B}")

text_sim = clip_text_similarity(
    roundtrip_prompt_A,
    prompt_B
)

image_sim = clip_image_similarity(
    image_A,
    image_C
)

print(f"\nText similarity  (A vs B): {text_sim:.3f}")
print(f"Image similarity (A vs C): {image_sim:.3f}")
print("(1.0 = identical embedding, 0.0 = unrelated)")

In [ ]:
free_memory("caption_model", "caption_processor", "sd_pipe")
print("Captioning model + SD1.5 pipeline freed ✅")

## 5. Inpainting: Editing *Only* Part of an Image 🖌️

**Task:** given an image, a **mask** (white = edit this, black = leave alone), and a new prompt, regenerate *just* the masked region while keeping everything outside it untouched. This is the technology behind "Magic Eraser" and "Generative Fill" style features.

We'll paint a simple circular mask over the center of one of our generated images and ask the model to replace whatever's there.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

INPAINT_ID =  # Write your modern ai name here
inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    INPAINT_ID, torch_dtype=torch.float16, safety_checker=None
).to(device)
inpaint_pipe.set_progress_bar_config(disable=True)
print("Loaded:", INPAINT_ID)

In [ ]:
base_image = image_A.resize((512, 512))

# A simple circular mask in the center — white = "please repaint this area"
mask = Image.new("L", (512, 512), 0)
draw = ImageDraw.Draw(mask)
draw.ellipse((130, 130, 382, 382), fill=255)

inpaint_prompt = "a golden trophy cup, studio lighting, highly detailed"
generator = torch.Generator(device=device).manual_seed(SEED)
inpainted = inpaint_pipe(
    prompt=inpaint_prompt,
    image=base_image,
    mask_image=mask,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images[0]

show_images(
    [base_image, mask.convert("RGB"), inpainted],
    titles=["Original", "Mask (white = edit)", f"Inpainted: '{inpaint_prompt[:25]}...'"],
    cols=3,
)

### 5.1 Did it edit *only* what we asked? A sanity-check metric

A good inpaint should (a) match the new prompt **inside** the mask, and (b) barely change anything **outside** the mask. We can check both:

- **CLIP score inside the mask region** — crop the masked area and score it against the new prompt.
- **Pixel MSE outside the mask** — compare original vs. inpainted pixels only where the mask is black; a good pipeline keeps this very low.

In [ ]:
mask_np = np.array(mask) > 127
orig_np = np.array(base_image).astype(np.float32)
inpainted_np = np.array(inpainted).astype(np.float32)

outside_mse = np.mean((orig_np[~mask_np] - inpainted_np[~mask_np]) ** 2)

# Crop a bounding box around the masked region for a focused CLIP score
ys, xs = np.where(mask_np)
crop = inpainted.crop((xs.min(), ys.min(), xs.max(), ys.max()))
inside_clip = clip_score(crop, inpaint_prompt)

print(f"Outside-mask pixel MSE (lower = background preserved better): {outside_mse:.2f}")
print(f"Inside-mask CLIP score vs '{inpaint_prompt}':                {inside_clip:.2f}")

In [ ]:
free_memory("inpaint_pipe")
print("Inpainting pipeline freed ✅")

## 6. Beyond Classic Diffusion: One-Step "Turbo" Generation ⚡

Everything in Section 2 needed **25–30 denoising steps**. That's the whole reason diffusion feels "slow" compared to, say, a single forward pass through a classifier.

The biggest practical trend in modern generative AI (2024–2026) is **distillation**: train a *student* model to jump straight from noise to a finished image in **1–4 steps**, by having it mimic what a slow teacher model would have produced after 25+ steps. Two well-known approaches:

- **Latent Consistency Models (LCM)** — trained so that any point along the noise-to-image trajectory maps to (approximately) the same final image.
- **Adversarial Diffusion Distillation (ADD)** — used by **SD-Turbo**, combines distillation with an adversarial loss so 1-step samples still look sharp.

Let's load **`stabilityai/sd-turbo`** and generate with just 1–4 steps, then put it head-to-head against Section 2's 25-step run: same idea (diffusion), radically different speed.

In [ ]:
from diffusers import AutoPipelineForText2Image

TURBO_ID =  # Write your modern ai name here
turbo_pipe = AutoPipelineForText2Image.from_pretrained(
    TURBO_ID, torch_dtype=torch.float16, safety_checker=None
).to(device)
turbo_pipe.set_progress_bar_config(disable=True)
print("Loaded:", TURBO_ID)

In [ ]:
turbo_prompt = sweep_prompt  # same prompt as our Section 2 hyperparameter sweep, for a fair comparison
turbo_steps_list = [1, 2, 4]

turbo_records, turbo_images, turbo_titles = [], [], []
for steps in turbo_steps_list:
    generator = torch.Generator(device=device).manual_seed(SEED)
    t0 = time.time()
    # SD-Turbo is distilled for guidance_scale=0.0 — it doesn't need classifier-free guidance
    img = turbo_pipe(turbo_prompt, num_inference_steps=steps, guidance_scale=0.0, generator=generator).images[0]
    elapsed = time.time() - t0
    turbo_images.append(img)
    turbo_titles.append(f"{steps} step(s), {elapsed:.2f}s")
    turbo_records.append({"model": "SD-Turbo", "steps": steps, "seconds": elapsed, "clip_score": clip_score(img, turbo_prompt)})

show_images(turbo_images, titles=turbo_titles, cols=len(turbo_steps_list))

### 6.1 Classic vs Turbo: speed and quality side by side

In [ ]:
classic_best = sweep_df[sweep_df.guidance_scale == 7.5].sort_values("steps").iloc[-1]
comparison = pd.DataFrame(
    [
        {"model": "Stable Diffusion v1.5 (classic)", "steps": int(classic_best.steps),
         "seconds": classic_best.seconds, "clip_score": classic_best.clip_score},
    ] + turbo_records
)[["model", "steps", "seconds", "clip_score"]]

comparison["images_per_second"] = 1 / comparison["seconds"]
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(comparison["model"] + "\n(" + comparison["steps"].astype(str) + " steps)", comparison["seconds"], color=["#4C72B0", "#DD8452", "#DD8452", "#DD8452"])
axes[0].set_ylabel("seconds per image"); axes[0].set_title("Speed"); axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(comparison["model"] + "\n(" + comparison["steps"].astype(str) + " steps)", comparison["clip_score"], color=["#4C72B0", "#DD8452", "#DD8452", "#DD8452"])
axes[1].set_ylabel("CLIP score"); axes[1].set_title("Prompt alignment"); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print("The trade-off in one line: Turbo trades a little CLIP score for a lot of speed —")
print("this is exactly the kind of trade-off that made on-device and real-time generative AI possible.")

In [ ]:
free_memory("turbo_pipe", "clip_model", "clip_processor")
print("All models freed ✅")

## 7. Wrap-Up 🎓

| Task | Model | Hugging Face ID | Output | Key metric |
|---|---|---|---|---|
| Text generation (LLM) | Qwen2.5-0.5B-Instruct | `Qwen/Qwen2.5-0.5B-Instruct` | token sequence | perplexity / human judgment |
| Text → Image | Stable Diffusion v1.5 | `stable-diffusion-v1-5/stable-diffusion-v1-5` | image | CLIP score |
| Image → Text | BLIP | `Salesforce/blip-image-captioning-base` | caption | CLIP text-image similarity |
| Inpainting | SD v1.5 Inpainting | `stable-diffusion-v1-5/stable-diffusion-inpainting` | edited image | CLIP score (inside) + pixel MSE (outside) |
| Fast / modern generation | SD-Turbo | `stabilityai/sd-turbo` | image | CLIP score + wall-clock time |

**The one pattern that repeats everywhere in `diffusers`:**
```python
pipe = AutoPipelineForText2Image.from_pretrained("<model-id>", torch_dtype=torch.float16).to("cuda")
image = pipe(prompt, num_inference_steps=<n>, guidance_scale=<g>).images[0]
```
Swap the model ID and you swap the *entire generative capability* — classic diffusion, inpainting, or a 1-step turbo model all speak the same API. Once that pattern clicks, exploring the frontier is mostly a matter of trying a new `model_id`.

### 🔭 Where this goes next (2026 and beyond)
- **Flow matching / rectified flow** (used in SD3, FLUX): a different mathematical recipe than classic denoising diffusion, trained to learn a *straight-line* path from noise to data instead of a curved one — fewer steps needed by construction, not just by distillation.
- **Unified multimodal models**: a single model that natively does text ↔ image ↔ audio without stitching two separate models together like our Section 4 round trip did.
- **Consistency & few-step everything**: the "distill it down to 1–4 steps" trick from Section 6 is being applied far beyond images — video, audio, and 3D generation are all racing toward real-time.

---
*Thank you for a great semester — from reading numbers (Week 1) to generating art in four steps (Week 12), you've now touched the full span of modern AI. 🎓✨*

---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 12 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>